# BitsAndBytes 4-bit/8-bit quantization

Use a CUDA Colab or compatible Linux GPU. BitsAndBytes is not claimed to work on macOS/MPS. The notebook compares model footprint, latency, and output for normal and quantized loading.

In [ ]:
%pip install -q "transformers==4.57.6" "accelerate==1.14.0" "bitsandbytes==0.50.0"

In [ ]:
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError(
        "Select a CUDA GPU runtime. CPU/MPS cannot execute this BitsAndBytes comparison."
    )
model_name = "facebook/opt-125m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
prompt = "Evidence-grounded answers should"

In [ ]:
configs = {
    "normal": None,
    "8bit": BitsAndBytesConfig(load_in_8bit=True),
    "4bit": BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    ),
}
results = {}
for name, quantization_config in configs.items():
    try:
        kwargs = {"device_map": "auto"}
        if quantization_config is not None:
            kwargs["quantization_config"] = quantization_config
        model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        torch.cuda.synchronize()
        started = time.perf_counter()
        output = model.generate(**inputs, max_new_tokens=24, do_sample=False)
        torch.cuda.synchronize()
        results[name] = {
            "memory_mb": model.get_memory_footprint() / 1024**2,
            "latency_seconds": time.perf_counter() - started,
            "output": tokenizer.decode(output[0], skip_special_tokens=True),
        }
        del model
        torch.cuda.empty_cache()
    except Exception as exc:  # noqa: BLE001 - compare unsupported modes safely
        results[name] = {"error": f"{type(exc).__name__}: {exc}"}
results

## Expected output

A dictionary reports footprint, generation latency, and output for each successful mode. Quantized footprints should be smaller, but latency and answer quality depend on hardware and model. Errors are retained rather than misreported as successful quantization.